## ETL of GeoSpatial data from US Census to Hugging Face

The US Census has restrictions on how it serves files.  So we'll copy them elsewhere first.


## Setup


In [4]:
%%capture
%%bash
apt-get update
apt-get install -y tree jq


In [5]:
import os
import duckdb
import pandas as pd
from google.colab import userdata


## Download all shapefiles


In [2]:
%%bash

hf download rwcitek/DUC-2026 \
  --repo-type dataset \
  --include "TIGER2024/**" \
  --local-dir ./


✓ Downloaded
  path: /content/TIGER2024


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Fetching 3234 files: 100%|██████████| 3234/3234 [07:34<00:00,  7.12it/s]


In [3]:
! find TIGER2024/ | grep -c zip$


3234


## Convert shapefiles to parquet




### State



In [9]:
zip_path = "TIGER2024/STATE/tl_2024_us_state.zip"
out_path = "parquet/TIGER2024/STATE"
out_parquet = f"{out_path}/tl_2024_us_state.parquet"


In [11]:
! mkdir -p {out_path}


In [7]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")


In [12]:
con.execute(
    f"""
    COPY (
        SELECT *
        FROM ST_Read('/vsizip/{zip_path}')
    )
    TO '{out_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """
)

In [14]:
! find {out_path} -ls


     4589      4 drwxr-xr-x   2 root     root         4096 Nov 21 22:06 parquet/TIGER2024/STATE
  6291589  11076 -rw-r--r--   1 root     root     11340375 Nov 21 22:06 parquet/TIGER2024/STATE/tl_2024_us_state.parquet


### Roads


#### Get a list of all shape files


In [32]:
roads_shapefiles = ! find ./TIGER2024/ROADS -name '*.zip' | sort
len(roads_shapefiles)


3233

In [33]:
roads_columns = []
for i, road in enumerate(roads_shapefiles[:]):
  if i % 100 == 0 : print(i, road)
  df = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM ST_Read('/vsizip/{road}')
  """).df()
  roads_columns.append([road, df["column_name"].values.tolist() ])

len(roads_columns)


0 ./TIGER2024/ROADS/tl_2024_01001_roads.zip
100 ./TIGER2024/ROADS/tl_2024_04007_roads.zip
200 ./TIGER2024/ROADS/tl_2024_06027_roads.zip


IOException: IO Error: GDAL Error (4): `/vsizip/./TIGER2024/ROADS/tl_2024_06079_roads.zip' does not exist in the file system, and is not recognized as a supported dataset name.

In [35]:
roads_df = pd.DataFrame( roads_columns )
roads_df


,0,1
0,./TIGER2024/ROADS/tl_2024_01001_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
1,./TIGER2024/ROADS/tl_2024_01003_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
2,./TIGER2024/ROADS/tl_2024_01005_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
3,./TIGER2024/ROADS/tl_2024_01007_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
4,./TIGER2024/ROADS/tl_2024_01009_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
...,...,...
221,./TIGER2024/ROADS/tl_2024_06069_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
222,./TIGER2024/ROADS/tl_2024_06071_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
223,./TIGER2024/ROADS/tl_2024_06073_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"
224,./TIGER2024/ROADS/tl_2024_06075_roads.zip,"[LINEARID, FULLNAME, RTTYP, MTFCC, geom]"


In [37]:
duckdb.query(f'''
  select "1", count("1")
  from roads_df
  group by all

''')

┌──────────────────────────────────────────┬────────────┐
│                    1                     │ count("1") │
│                varchar[]                 │   int64    │
├──────────────────────────────────────────┼────────────┤
│ [LINEARID, FULLNAME, RTTYP, MTFCC, geom] │        226 │
└──────────────────────────────────────────┴────────────┘

In [ ]:
# Wait for roads downloads to finish
%%capture
%%bash

while [ -f /tmp/tiger.roads.sh ] ; do
  date
  sleep 1
done


### Verify


In [ ]:
! tree TIGER2024/


In [ ]:
! find ./TIGER2024/ | grep -c zip$


# Keep


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = "TIGER2024/"
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


In [ ]:
%%capture hf_repo_list
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf datasets  \
  list -R \
  "${TARGET_REPO}"/ \


In [ ]:
print(hf_repo_list.stdout)
